<a href="https://colab.research.google.com/github/somaiah-ui/CrewAI-Project/blob/main/4-Agent_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U crewai litellm tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 892.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
# ================================================================
# 🤖 CREWAI WEB RESEARCH ASSISTANT
#
# Google Colab Version
# CrewAI + Gemini + Tavily
#
# FEATURES:
# ✅ Async execution for Colab
# ✅ Gemini model fallback
# ✅ Automatic retries
# ✅ Handles 503 server overload
# ✅ Handles 429 rate limits
# ✅ Handles unavailable models
# ✅ Tavily error handling
# ✅ API key validation
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import asyncio
import random

from getpass import getpass

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

from tavily import TavilyClient


# ================================================================
# TITLE
# ================================================================

print("\n" + "=" * 70)
print("🤖 CREWAI WEB RESEARCH ASSISTANT")
print("=" * 70)


# ================================================================
# API KEYS
# ================================================================

GEMINI_API_KEY = getpass(
    "\n🔑 Enter your Gemini API key: "
).strip()

TAVILY_API_KEY = getpass(
    "🔑 Enter your Tavily API key: "
).strip()


# Check that keys were actually entered

if not GEMINI_API_KEY:
    raise ValueError(
        "❌ Gemini API key was not entered."
    )

if not TAVILY_API_KEY:
    raise ValueError(
        "❌ Tavily API key was not entered."
    )


# ================================================================
# TAVILY CLIENT
# ================================================================

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY
)


# ================================================================
# TAVILY WEB SEARCH TOOL
# ================================================================

@tool("Tavily Web Search")
def web_search(query: str) -> str:
    """
    Search the live internet using Tavily.

    Use this tool whenever current or factual information
    about a topic is required.
    """

    print(f"\n🌐 Searching Tavily: {query}")

    try:

        response = tavily_client.search(
            query=query,
            search_depth="basic",
            max_results=5
        )

        results = response.get("results", [])


        # --------------------------------------------------------
        # NO RESULTS
        # --------------------------------------------------------

        if not results:

            return """
No useful search results were returned.

Try searching the topic using a different query.
"""


        # --------------------------------------------------------
        # FORMAT RESULTS
        # --------------------------------------------------------

        formatted_results = []

        for number, result in enumerate(
            results,
            start=1
        ):

            title = result.get(
                "title",
                "No title"
            )

            content = result.get(
                "content",
                "No description available."
            )

            url = result.get(
                "url",
                "No URL available."
            )

            formatted_results.append(
                f"""
SEARCH RESULT {number}

Title:
{title}

Information:
{content}

Source:
{url}

------------------------------------------------------------
"""
            )


        return "\n".join(
            formatted_results
        )


    # ------------------------------------------------------------
    # HANDLE TAVILY ERRORS
    # ------------------------------------------------------------

    except Exception as error:

        error_text = str(error)

        print(
            f"\n⚠️ Tavily search problem: {error_text}"
        )


        # Bad API key

        if (
            "401" in error_text
            or "403" in error_text
            or "unauthorized" in error_text.lower()
            or "api key" in error_text.lower()
        ):

            return """
SEARCH ERROR:

The Tavily API key appears to be invalid.

The agent should not invent current information.
"""


        # Rate limit

        if (
            "429" in error_text
            or "rate limit" in error_text.lower()
        ):

            return """
SEARCH ERROR:

Tavily temporarily reached its request limit.

Use available information and clearly state that
web search was temporarily unavailable.
"""


        # Other Tavily error

        return f"""
SEARCH ERROR:

Tavily could not complete the search.

Technical information:
{error_text}

Do not invent information that could not be verified.
"""


# ================================================================
# GEMINI MODEL FALLBACK LIST
# ================================================================
#
# We DO NOT rely only on Gemini 3.8 Flash.
#
# If Google says one model is overloaded, the program can
# automatically try another currently supported model.
#
# ================================================================

MODEL_LIST = [

    # Fast/lightweight model first
    "gemini/gemini-3.5-flash-lite",

    # More capable fallback
    "gemini/gemini-3.5-flash",

    # Additional stable fallbacks
    "gemini/gemini-3.6-flash",
    "gemini/gemini-3.7-flash",

    # Most powerful Flash model as final fallback
    "gemini/gemini-3.8-flash"
]


# ================================================================
# FUNCTION TO BUILD THE CREW
# ================================================================

def build_crew(model_name, topic):


    # ============================================================
    # CREATE LLM
    # ============================================================

    llm = LLM(

        model=model_name,

        api_key=GEMINI_API_KEY
    )


    # ============================================================
    # AGENT 1
    # INTERNET RESEARCHER
    # ============================================================

    researcher = Agent(

        role="Internet Researcher",

        goal=f"""
Research accurate and useful information about:

{topic}

Use current internet information whenever necessary.
""",

        backstory="""
You are a careful internet research specialist.

You search for reliable information and organize
it into useful notes.

You focus on:

- definitions
- important facts
- explanations
- examples
- advantages
- disadvantages
- practical applications

Never invent facts.

If current information is needed, use the
Tavily Web Search tool.
""",

        tools=[
            web_search
        ],

        llm=llm,

        verbose=True,

        allow_delegation=False
    )


    # ============================================================
    # AGENT 2
    # TEACHER
    # ============================================================

    teacher = Agent(

        role="Friendly Teacher",

        goal=f"""
Explain the topic '{topic}' in simple language
using the research produced by the researcher.
""",

        backstory="""
You are a friendly and experienced teacher.

Your specialty is converting complicated
information into explanations that students
can easily understand.

Use:

- simple language
- short explanations
- bullet points
- practical examples

Do not unnecessarily repeat information.

Do not invent facts that were not supported
by the research.
""",

        llm=llm,

        verbose=True,

        allow_delegation=False
    )


    # ============================================================
    # TASK 1
    # RESEARCH
    # ============================================================

    research_task = Task(

        description=f"""
Research the following topic:

{topic}


Use Tavily Web Search whenever useful.


Find information about:

1. Definition

2. Important concepts

3. How it works

4. Important facts

5. Simple examples

6. Advantages

7. Disadvantages

8. Real-world applications

9. Recent developments if relevant


IMPORTANT RULES:

- Do not invent statistics.

- Do not invent sources.

- Do not invent companies or technologies.

- Include URLs for important information.

- If web search fails, clearly mention that instead
  of making up information.
""",

        expected_output="""
A concise research report containing:

- Definition
- Important concepts
- Explanation
- Examples
- Advantages
- Disadvantages
- Applications
- Relevant sources
""",

        agent=researcher
    )


    # ============================================================
    # TASK 2
    # TEACHING
    # ============================================================

    teaching_task = Task(

        description=f"""
Teach the student about:

{topic}


Use the research generated by the
Internet Researcher.


Create the explanation using this format:


1. WHAT IS IT?

Explain the topic in simple words.


2. HOW DOES IT WORK?

Explain the basic working.


3. SIMPLE EXAMPLE

Give an easy-to-understand example.


4. IMPORTANT POINTS

Give the most important things to remember.


5. ADVANTAGES

Explain important advantages.


6. DISADVANTAGES

Explain important disadvantages.


7. REAL-WORLD APPLICATIONS

Give practical examples.


8. QUICK SUMMARY

Summarize the topic in a few sentences.


Keep the answer concise and student-friendly.
""",

        expected_output="""
A clear and easy-to-understand explanation
of the topic based on the research.
""",

        agent=teacher,

        context=[
            research_task
        ]
    )


    # ============================================================
    # CREATE CREW
    # ============================================================

    crew = Crew(

        agents=[
            researcher,
            teacher
        ],

        tasks=[
            research_task,
            teaching_task
        ],

        process=Process.sequential,

        verbose=True
    )


    return crew


# ================================================================
# ERROR CLASSIFICATION
# ================================================================

def classify_error(error):

    text = str(error).lower()


    # ------------------------------------------------------------
    # SERVER BUSY / TEMPORARY ERROR
    # ------------------------------------------------------------

    if (
        "503" in text
        or "unavailable" in text
        or "high demand" in text
        or "temporarily unavailable" in text
    ):

        return "temporary"


    # ------------------------------------------------------------
    # RATE LIMIT
    # ------------------------------------------------------------

    if (
        "429" in text
        or "resource_exhausted" in text
        or "rate limit" in text
        or "quota" in text
    ):

        return "rate_limit"


    # ------------------------------------------------------------
    # MODEL DOES NOT EXIST / NOT AVAILABLE
    # ------------------------------------------------------------

    if (
        "404" in text
        or "not_found" in text
        or "model not found" in text
        or "no longer available" in text
    ):

        return "model"


    # ------------------------------------------------------------
    # API KEY / AUTH ERROR
    # ------------------------------------------------------------

    if (
        "401" in text
        or "403" in text
        or "api_key_invalid" in text
        or "invalid api key" in text
        or "permission_denied" in text
    ):

        return "authentication"


    # ------------------------------------------------------------
    # NETWORK / TIMEOUT
    # ------------------------------------------------------------

    if (
        "timeout" in text
        or "timed out" in text
        or "connection" in text
        or "network" in text
    ):

        return "network"


    return "unknown"


# ================================================================
# RUN CREW WITH AUTOMATIC RETRIES + MODEL FALLBACK
# ================================================================

async def run_crew_safely(topic):

    print("\n" + "=" * 70)
    print("🚀 STARTING CREWAI")
    print("=" * 70)


    # ------------------------------------------------------------
    # TRY EACH GEMINI MODEL
    # ------------------------------------------------------------

    for model_number, model_name in enumerate(
        MODEL_LIST,
        start=1
    ):

        print(
            f"\n🤖 Trying model {model_number}/{len(MODEL_LIST)}"
        )

        print(
            f"Model: {model_name}"
        )


        # --------------------------------------------------------
        # RETRY EACH MODEL UP TO 2 TIMES
        # --------------------------------------------------------

        for attempt in range(1, 3):

            try:

                print(
                    f"\n▶ Attempt {attempt}/2"
                )


                # Fresh crew for every attempt
                crew = build_crew(
                    model_name,
                    topic
                )


                # =================================================
                # IMPORTANT:
                #
                # GOOGLE COLAB ALREADY HAS AN EVENT LOOP.
                #
                # Therefore we MUST use kickoff_async().
                # =================================================

                result = await crew.kickoff_async()


                # -------------------------------------------------
                # SUCCESS
                # -------------------------------------------------

                print(
                    f"\n✅ Success using {model_name}"
                )

                return result


            # =====================================================
            # ERROR HANDLING
            # =====================================================

            except Exception as error:

                error_type = classify_error(
                    error
                )

                error_message = str(error)


                print(
                    "\n⚠️ CrewAI encountered a problem."
                )

                print(
                    f"Type: {error_type}"
                )


                # -------------------------------------------------
                # INVALID GEMINI API KEY
                # -------------------------------------------------

                if error_type == "authentication":

                    raise RuntimeError(
                        """
❌ GEMINI AUTHENTICATION FAILED

Your Gemini API key appears to be invalid,
expired, restricted, or does not have access.

Create/check the key in Google AI Studio and
run this cell again with the correct key.
"""
                    ) from error


                # -------------------------------------------------
                # MODEL NOT AVAILABLE
                # -------------------------------------------------

                elif error_type == "model":

                    print(
                        "⚠️ This Gemini model is not available."
                    )

                    print(
                        "➡️ Switching to the next model..."
                    )

                    break


                # -------------------------------------------------
                # TEMPORARY 503
                # -------------------------------------------------

                elif error_type == "temporary":

                    print(
                        "⏳ Gemini is temporarily overloaded."
                    )


                    if attempt < 2:

                        # Wait before retrying

                        delay = 5 + random.randint(
                            1,
                            3
                        )

                        print(
                            f"🔄 Retrying in {delay} seconds..."
                        )

                        await asyncio.sleep(
                            delay
                        )

                        continue


                    else:

                        print(
                            "➡️ Switching to another Gemini model..."
                        )

                        break


                # -------------------------------------------------
                # RATE LIMIT / QUOTA
                # -------------------------------------------------

                elif error_type == "rate_limit":

                    if attempt < 2:

                        delay = 10

                        print(
                            f"⏳ Rate limit detected."
                        )

                        print(
                            f"🔄 Waiting {delay} seconds..."
                        )

                        await asyncio.sleep(
                            delay
                        )

                        continue


                    print(
                        "➡️ Trying another model..."
                    )

                    break


                # -------------------------------------------------
                # NETWORK ERROR
                # -------------------------------------------------

                elif error_type == "network":

                    if attempt < 2:

                        print(
                            "🌐 Temporary network issue."
                        )

                        print(
                            "🔄 Retrying in 5 seconds..."
                        )

                        await asyncio.sleep(
                            5
                        )

                        continue


                    print(
                        "➡️ Trying another model..."
                    )

                    break


                # -------------------------------------------------
                # UNKNOWN ERROR
                # -------------------------------------------------

                else:

                    print(
                        "\n❌ Unexpected error:"
                    )

                    print(
                        error_message
                    )

                    raise


    # ============================================================
    # IF ALL MODELS FAIL
    # ============================================================

    raise RuntimeError(
        """
❌ ALL GEMINI MODELS FAILED

Possible causes:

• Google Gemini servers are temporarily busy
• Your Gemini API quota has been reached
• Your API key does not have model access
• Temporary internet/API outage

Wait a few minutes and run Cell 2 again.
"""
    )


# ================================================================
# USER INPUT
# ================================================================

print("\n" + "=" * 70)
print("📚 ENTER YOUR TOPIC")
print("=" * 70)

topic = input(
    "\nWhat would you like to research?\n\n> "
).strip()


if not topic:

    raise ValueError(
        "❌ Please enter a topic."
    )


# ================================================================
# SHOW WORKFLOW
# ================================================================

print(
    """

CREW WORKFLOW

👤 User Topic
      ↓
🌐 Research Agent
      ↓
🔎 Tavily Web Search
      ↓
📚 Research Notes
      ↓
🧑‍🏫 Teacher Agent
      ↓
✅ Final Explanation

"""
)


# ================================================================
# RUN
# ================================================================

result = await run_crew_safely(
    topic
)


# ================================================================
# FINAL RESULT
# ================================================================

print("\n\n")

print("=" * 70)
print("✅ FINAL ANSWER")
print("=" * 70)

print("\n")

print(result)


🤖 CREWAI WEB RESEARCH ASSISTANT

🔑 Enter your Gemini API key: ··········
🔑 Enter your Tavily API key: ··········

📚 ENTER YOUR TOPIC

What would you like to research?

> What is Agentic AI?


CREW WORKFLOW

👤 User Topic
      ↓
🌐 Research Agent
      ↓
🔎 Tavily Web Search
      ↓
📚 Research Notes
      ↓
🧑‍🏫 Teacher Agent
      ↓
✅ Final Explanation



🚀 STARTING CREWAI

🤖 Trying model 1/5
Model: gemini/gemini-3.5-flash-lite

▶ Attempt 1/2


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ac17b7ac-2912-4d28-8c7c-046b0a6794b5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Research the following topic:                                                                                  │
│                                                                                                                 │
│  What is Agentic AI?                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  Use Tavily Web Search whenever useful.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Find information about:                                                                                        │
│                                                                                                                 │
│  1. Definition                                                                                                  │
│                                                                                                                 │
│  2. Important concepts                                                                                          │
│                                                                                                                 │
│  3. How it works                                                                                                │
│                                                                                                                 │
│  4. Important facts                                                                                             │
│                                                                                                                 │
│  5. Simple examples                                                                                             │
│                                                                                                                 │
│  6. Advantages                                                                                                  │
│                                                                                                                 │
│  7. Disadvantages                                                                                               │
│                                                                                                                 │
│  8. Real-world applications                                                                                     │
│                                                                                                                 │
│  9. Recent developments if relevant                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  IMPORTANT RULES:                                                                                               │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Research the following topic:                                                                                  │
│                                                                                                                 │
│  What is Agentic AI?                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  Use Tavily Web Search whenever useful.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Find information about:                                                                                        │
│                                                                                                                 │
│  1. Definition                                                                                                  │
│                                                                                                                 │
│  2. Important concepts                                                                                          │
│                                                                                                                 │
│  3. How it works                                                                                                │
│                                                                                                                 │
│  4. Important facts                                                                                             │
│                                                                                                                 │
│  5. Simple examples                                                                                             │
│                                                                                                                 │
│  6. Advantages                                                                                                  │
│                                                                                                                 │
│  7. Disadvantages                                                                                               │
│                                                                                                                 │
│  8. Real-world applications                                                                                     │
│                                                                                                                 │
│  9. Recent developments if relevant                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  IMPORTANT RULES:                                      


🌐 Searching Tavily: Agentic AI definition concepts examples applications


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Friendly Teacher                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello! I am so glad you are here to learn. Today, we are going to look at a very exciting topic called         │
│  **Agentic AI**. Let's break it down into easy pieces!                                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. WHAT IS IT?                                                                                             │
│  * Agentic AI is an advanced type of artificial intelligence.                                                   │
│  * Unlike regular AI that just waits for your prompt and answers one step at a time, Agentic AI can think,      │
│  make its own choices, and work toward a big goal all by itself.                                                │
│                                                                                                                 │
│  ### 2. HOW DOES IT WORK?                                                                                       │
│  Agentic AI follows a loop to get things done:                                                                  │
│  * **Perceive:** It looks at its environment or gets a big goal.                                                │
│  * **Plan:** It breaks that big goal down into smaller, easy steps.                                             │
│  * **Act:** It uses external tools, software, or APIs to do the work.                                           │
│  * **Self-Correct:** It checks its own results, fixes its mistakes, and tries again until the job is done.      │
│                                                                                                                 │
│  ### 3. SIMPLE EXAMPLE                                                                                          │
│  * Imagine you tell an AI: *"Increase my website sign-ups by 10%."*                                             │
│  * Instead of asking you what to do next, the AI creates the ads, watches how they perform, shifts the budget   │
│  around, and fixes the ads if they aren't working—all without you needing to step in.                           │
│                                                                                                                 │
│  ### 4. IMPORTANT POINTS                                                                                        │
│  * **Autonomy:** It works by itself without constant human direction.                                           │
│  * **Tool Use:** It can use external tools, databases, and APIs.                                                │
│  * **Action-Oriented:** It focuses on finishing whole workflows, not just generating text or images.            │
│                                                                                                                 │
│  ### 5. ADVANTAGES                                                                                              │
│  * **Saves Time:** It handles multi-step tasks so humans don't have to.                                         │
│  * **Adapts Easily:** Unlike older software, it can adj

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Teach the student about:                                                                                       │
│                                                                                                                 │
│  What is Agentic AI?                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  Use the research generated by the                                                                              │
│  Internet Researcher.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  Create the explanation using this format:                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  1. WHAT IS IT?                                                                                                 │
│                                                                                                                 │
│  Explain the topic in simple words.                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  2. HOW DOES IT WORK?                                                                                           │
│                                                                                                                 │
│  Explain the basic working.                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  3. SIMPLE EXAMPLE                                                                                              │
│                                                                                                                 │
│  Give an easy-to-understand example.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  4. IMPORTANT POINTS                                                                                            │
│                                                                                                                 │
│  Give the most important things to remember.                                                                    │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ac17b7ac-2912-4d28-8c7c-046b0a6794b5                                                                       │
│  Final Output: Hello! I am so glad you are here to learn. Today, we are going to look at a very exciting topic  │
│  called **Agentic AI**. Let's break it down into easy pieces!                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. WHAT IS IT?                                                                                             │
│  * Agentic AI is an advanced type of artificial intelligence.                                                   │
│  * Unlike regular AI that just waits for your prompt and answers one step at a time, Agentic AI can think,      │
│  make its own choices, and work toward a big goal all by itself.                                                │
│                                                                                                                 │
│  ### 2. HOW DOES IT WORK?                                                                                       │
│  Agentic AI follows a loop to get things done:                                                                  │
│  * **Perceive:** It looks at its environment or gets a big goal.                                                │
│  * **Plan:** It breaks that big goal down into smaller, easy steps.                                             │
│  * **Act:** It uses external tools, software, or APIs to do the work.                                           │
│  * **Self-Correct:** It checks its own results, fixes its mistakes, and tries again until the job is done.      │
│                                                                                                                 │
│  ### 3. SIMPLE EXAMPLE                                                                                          │
│  * Imagine you tell an AI: *"Increase my website sign-ups by 10%."*                                             │
│  * Instead of asking you what to do next, the AI creates the ads, watches how they perform, shifts the budget   │
│  around, and fixes the ads if they aren't working—all without you needing to step in.                           │
│                                                                                                                 │
│  ### 4. IMPORTANT POINTS                                                                                        │
│  * **Autonomy:** It works by itself without constant human direction.                                           │
│  * **Tool Use:** It can use external tools, databases, and APIs.                                                │
│  * **Action-Oriented:** It focuses on finishing whole workflows, not just generating text or images.            │
│                                                                                                                 │
│  ### 5. ADVANTAGES                                                                                              │
│  * **Saves Time:** It handles multi-step tasks so humans don't have to.                                         │
│  * **Adapts Easily:** Unlike older software, it can ad


✅ Success using gemini/gemini-3.5-flash-lite



✅ FINAL ANSWER


Hello! I am so glad you are here to learn. Today, we are going to look at a very exciting topic called **Agentic AI**. Let's break it down into easy pieces!

---

### 1. WHAT IS IT?
* Agentic AI is an advanced type of artificial intelligence.
* Unlike regular AI that just waits for your prompt and answers one step at a time, Agentic AI can think, make its own choices, and work toward a big goal all by itself.

### 2. HOW DOES IT WORK?
Agentic AI follows a loop to get things done:
* **Perceive:** It looks at its environment or gets a big goal.
* **Plan:** It breaks that big goal down into smaller, easy steps.
* **Act:** It uses external tools, software, or APIs to do the work.
* **Self-Correct:** It checks its own results, fixes its mistakes, and tries again until the job is done.

### 3. SIMPLE EXAMPLE
* Imagine you tell an AI: *"Increase my website sign-ups by 10%."*
* Instead of asking you what to do next, the AI creat